In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
folder = '/content/drive/MyDrive/TransformesCod/03-restaurant/meal_names/'
file_path = '/content/drive/MyDrive/TransformesCod/03-restaurant/meal_names/meals_names.txt'
save_output_path = folder + 'Model'

In [ ]:
from transformers import GPT2Tokenizer , GPT2LMHeadModel

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, tokenizer, file_path, block_size=128):
        with open(file_path, encoding="utf-8") as f:
            text = f.read()

        tokenized_text = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text))

        self.examples = []
        for i in range(0, len(tokenized_text) - block_size + 1, block_size):
            self.examples.append(tokenized_text[i : i + block_size])

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return torch.tensor(self.examples[i], dtype=torch.long)

def load_dataset(file_path, tokenizer, block_size=128):
    dataset = TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=block_size
    )
    return dataset

train_dataset = load_dataset(file_path, tokenizer)
print(f"تم تحميل البيانات بنجاح! عدد الكتل: {len(train_dataset)}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

تم تحميل البيانات بنجاح! عدد الكتل: 3


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=save_output_path,
    num_train_epochs=32,
    per_device_train_batch_size=1,
    report_to="none"
)

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

In [ ]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=96, training_loss=1.1166973114013672, metrics={'train_runtime': 52.7052, 'train_samples_per_second': 1.821, 'train_steps_per_second': 1.821, 'total_flos': 6271008768000.0, 'train_loss': 1.1166973114013672, 'epoch': 32.0})

In [23]:
input_text = "the name of a new creative and funy meal is :"


input_ids = tokenizer.encode(input_text, return_tensors="pt")
n_meals = 10
#model = model.to('cpu')
model = trainer.model
model.eval()
output = model.generate(
    input_ids,
    max_length=20,
    num_return_sequences=n_meals,
    temperature=0.7,
    repetition_penalty=2.0,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

import re

def filter_non_english(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

input_text = "the name of a new creative and funy meal is"
for meal in output:

    decoded_output = tokenizer.decode(meal, skip_special_tokens=True)

    meal_name = decoded_output.split(input_text)[-1].split('.')[0].strip()

    meal_name = filter_non_english(meal_name)
    if len(meal_name) > 0:
        print(f"Meal Name: {meal_name}")

Meal Name: Chicken Caesar Wrap with Cabbage
Meal Name: Chicken Caesar Wrap with rice
Meal Name: P Chicken Parmesan with Garlic
Meal Name: P Lunch Time pm
Meal Name: Chicken Caesar Wrap recipe
Meal Name: D Chicken Parmesan Cabbage
Meal Name: You may have heard that the
Meal Name: p Chicken Parmesan I think
Meal Name: Chicken Parmesan I like
Meal Name: D Chicken Parmesan served with
